![image.png](https://i.imgur.com/a3uAqnb.png)

# 🚀 Object Detection with Faster R-CNN on COCO Dataset

In this lab, we will:

✅ Build a **custom Dataset class** for the **COCO 2017 dataset**  
✅ Use a **pretrained Faster R-CNN model** for object detection  
✅ Train and evaluate the model using **mAP@0.5:0.95**  
✅ Visualize predictions vs. ground truth  

---


## 1️⃣ Dataset Class

We use the **COCO 2017 dataset**, which contains images with **bounding boxes and category labels** — one of the richest object detection benchmarks available.  
PyTorch provides a built-in loader: `torchvision.datasets.CocoDetection`.

### 🔹 Install Dependencies


In [1]:
from IPython.display import clear_output
!pip install pycocotools torchmetrics
clear_output()
print("Dependencies installed.")


Dependencies installed.


### 🔹 Download COCO 2017 Validation Data (used for quick experimentation)

> **Note:** The full COCO train set is ~18 GB. For this lab we use the **val2017** split (~1 GB, 5,000 images) for both training demo and evaluation.  
> Replace with `train2017` for full training.


In [2]:
import os

os.makedirs("./COCO_data", exist_ok=True)

# Download val2017 images
!wget -q --show-progress -O ./COCO_data/val2017.zip http://images.cocodataset.org/zips/val2017.zip
!unzip -q ./COCO_data/val2017.zip -d ./COCO_data/

# Download annotations
!wget -q --show-progress -O ./COCO_data/annotations.zip http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q ./COCO_data/annotations.zip -d ./COCO_data/

print("Download complete!")


./COCO_data/val2017 100%[===================>] 777.80M  29.7MB/s    in 17s     
./COCO_data/annotat 100%[===================>] 241.19M  52.6MB/s    in 5.0s    
Download complete!


### 🔹 COCO Class Mappings

COCO has **80 object categories** (with non-contiguous IDs from 1–90).  
We remap them to contiguous 0-indexed labels for the model.


In [3]:
# COCO has 80 categories with non-contiguous IDs (1-90, some gaps).
# We build a mapping: coco_id -> contiguous 0-indexed label

COCO_CLASSES = {
    1: "person", 2: "bicycle", 3: "car", 4: "motorcycle", 5: "airplane",
    6: "bus", 7: "train", 8: "truck", 9: "boat", 10: "traffic light",
    11: "fire hydrant", 13: "stop sign", 14: "parking meter", 15: "bench",
    16: "bird", 17: "cat", 18: "dog", 19: "horse", 20: "sheep", 21: "cow",
    22: "elephant", 23: "bear", 24: "zebra", 25: "giraffe", 27: "backpack",
    28: "umbrella", 31: "handbag", 32: "tie", 33: "suitcase", 34: "frisbee",
    35: "skis", 36: "snowboard", 37: "sports ball", 38: "kite",
    39: "baseball bat", 40: "baseball glove", 41: "skateboard", 42: "surfboard",
    43: "tennis racket", 44: "bottle", 46: "wine glass", 47: "cup",
    48: "fork", 49: "knife", 50: "spoon", 51: "bowl", 52: "banana",
    53: "apple", 54: "sandwich", 55: "orange", 56: "broccoli", 57: "carrot",
    58: "hot dog", 59: "pizza", 60: "donut", 61: "cake", 62: "chair",
    63: "couch", 64: "potted plant", 65: "bed", 67: "dining table",
    70: "toilet", 72: "tv", 73: "laptop", 74: "mouse", 75: "remote",
    76: "keyboard", 77: "cell phone", 78: "microwave", 79: "oven",
    80: "toaster", 81: "sink", 82: "refrigerator", 84: "book",
    85: "clock", 86: "vase", 87: "scissors", 88: "teddy bear",
    89: "hair drier", 90: "toothbrush",
}

# Remap to contiguous 0-indexed labels
sorted_coco_ids = sorted(COCO_CLASSES.keys())
coco_id_to_label = {coco_id: idx for idx, coco_id in enumerate(sorted_coco_ids)}
label_to_coco_id = {v: k for k, v in coco_id_to_label.items()}
label_to_name = {coco_id_to_label[cid]: name for cid, name in COCO_CLASSES.items()}

NUM_CLASSES = len(COCO_CLASSES)  # 80
print(f"Total COCO categories: {NUM_CLASSES}")
print("Sample mappings:", {k: label_to_name[k] for k in range(5)})


Total COCO categories: 80
Sample mappings: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane'}


### 🔹 Custom Dataset Class with Proper Formatting

We wrap `CocoDetection` to:
- Convert **COCO [x, y, w, h]** bounding boxes → **[xmin, ymin, xmax, ymax]** (the format Faster R-CNN expects)
- Remap category IDs to our contiguous labels
- Filter out invalid / degenerate boxes


In [4]:
import torch
import torchvision
from torchvision.datasets import CocoDetection
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as T

class CocoDetectionDataset(CocoDetection):
    """
    Wraps torchvision.datasets.CocoDetection to return targets in
    Faster R-CNN format: dicts with 'boxes' and 'labels' tensors.
    """

    def __init__(self, img_folder, ann_file, transforms=None):
        super().__init__(img_folder, ann_file)
        self._transforms = transforms

    def __getitem__(self, idx):
        img, raw_targets = super().__getitem__(idx)

        if self._transforms:
            img = self._transforms(img)

        boxes, labels = [], []
        for ann in raw_targets:
            coco_cat = ann["category_id"]
            if coco_cat not in coco_id_to_label:
                continue  # Skip categories not in our mapping

            # COCO box format: [x_min, y_min, width, height]
            x, y, w, h = ann["bbox"]
            xmin, ymin, xmax, ymax = x, y, x + w, y + h

            # Skip degenerate boxes
            if xmax <= xmin or ymax <= ymin:
                continue

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(coco_id_to_label[coco_cat])

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32) if boxes
                     else torch.zeros((0, 4), dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64),
        }
        return img, target


def custom_collate_fn(batch):
    """Handle variable number of bounding boxes per image."""
    return tuple(zip(*batch))


# Paths (adjust if you placed data elsewhere)
IMG_DIR = "./COCO_data/val2017"
ANN_FILE = "./COCO_data/annotations/instances_val2017.json"

transform = T.Compose([T.ToTensor()])

full_dataset = CocoDetectionDataset(IMG_DIR, ANN_FILE, transforms=transform)

# Use a subset for quick experimentation (increase for real training)
TRAIN_SIZE, VAL_SIZE = 800, 200
train_dataset = Subset(full_dataset, list(range(TRAIN_SIZE)))
val_dataset   = Subset(full_dataset, list(range(TRAIN_SIZE, TRAIN_SIZE + VAL_SIZE)))

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True,
                          num_workers=2, collate_fn=custom_collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=2, shuffle=False,
                          num_workers=2, collate_fn=custom_collate_fn)

print(f"Train images : {len(train_dataset)}")
print(f"Val   images : {len(val_dataset)}")
print(f"Train batches: {len(train_loader)}")


loading annotations into memory...
Done (t=0.88s)
creating index...
index created!
Train images : 800
Val   images : 200
Train batches: 400


### 🔹 Why Do We Need a Custom `collate_fn`?

Unlike classification, where every image has a single label, detection images have **variable numbers of bounding boxes**.

- The default `torch.stack`-based collate **fails** because box tensors differ in size.
- Our custom function **zips** images and targets into tuples instead:

```
# Before: list of (image, target) pairs
[(img1, tgt1), (img2, tgt2), ...]

# After: two tuples
images  = (img1, img2, ...)
targets = (tgt1, tgt2, ...)
```


---

## 2️⃣ Model Class

We use a **pretrained Faster R-CNN with a MobileNetV3-Large FPN backbone**.

### 🔹 Why MobileNetV3?
- Lightweight and fast — suitable for fine-tuning on consumer hardware.
- Pre-trained on **COCO** (91 classes including background), so its backbone already "knows" COCO-like features.
- We only replace the **classification head** to match our 80-class remapped label space.


In [ ]:
import torchvision

# Load pretrained Faster R-CNN (backbone pretrained on COCO)
model = torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn(...)

# Replace the box predictor head: NUM_CLASSES + 1 (background class)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = (
   ...
)

# Freeze backbone — only fine-tune the detection head
... # requires_grad_(False)
... # roi_heads.box_predictor.requires_grad_(True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model on: {device}")
model


### 🔹 Backbone Freezing Strategy

| Approach | Trainable params | Training speed | Risk of overfitting |
|---|---|---|---|
| Freeze all backbone | Low | Fast | Low |
| Freeze backbone, train head ✅ | Medium | Moderate | Low |
| Train all layers | High | Slow | Higher |

We take the **middle path**: freeze the feature extractor, fine-tune only the head.  
This works well when the backbone is already trained on similar data (COCO → COCO).


---

## 3️⃣ Training and Validation Loops

### 🔹 Training Loop

Faster R-CNN computes all losses **internally** when targets are supplied.  
The loss dictionary contains four components:

| Key | Meaning |
|---|---|
| `loss_classifier` | Cross-entropy over RoI classes |
| `loss_box_reg` | Smooth-L1 regression loss for box deltas |
| `loss_objectness` | RPN objectness (foreground vs background) |
| `loss_rpn_box_reg` | RPN bounding-box regression loss |

We simply sum them and back-propagate.


In [6]:
from tqdm import tqdm

def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0

    for images, targets in tqdm(dataloader, desc="Training"):
        # Move images to device, loop over all images
        images = ...

        # Move targets to device
        # Hint: target strcuture targets = {"boxes": [....], "labels": [....] }
        targets =


        # Skip batches where all images have no annotations
        if all(len(t["boxes"]) == 0 for t in targets):
            continue

        # Forward pass — returns loss dict in train mode
        loss_dict = model(..., ... )
        losses = ... # sum all losses in loss_dict

        # optimizer update
        ...
        ...
        ...

        total_loss += losses.item()

    return total_loss / len(dataloader)


### 🔹 Validation Loop

#### 📌 Intersection over Union (IoU)

A predicted box is **correct** when it overlaps enough with a ground-truth box.  
We measure this with **IoU**:

$$IoU = \frac{\text{Area of Overlap}}{\text{Area of Union}}$$

🚀 **Higher IoU = Better localisation!**

#### 📌 mAP@0.5:0.95

**mAP** (mean Average Precision) is the standard detection metric.  
**mAP@0.5:0.95** averages precision across IoU thresholds from 0.5 to 0.95 in 0.05 steps:

| IoU threshold | Interpretation |
|---|---|
| 0.50 | Loose match (PASCAL VOC style) |
| 0.75 | Strict match |
| 0.95 | Very strict match |

A **single number** summarising performance across all difficulty levels.


In [7]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

metric = MeanAveragePrecision(
    iou_thresholds=[0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
)

def validate(model, dataloader, device):
    """Evaluate with mAP@0.5:0.95."""
    model.eval()
    # reset the matric
    ...

    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Validation"):
            images = ...
            preds  = ...

            processed_preds = [
                {
                    "boxes":  p["boxes"].cpu(),
                    "scores": p["scores"].cpu(),
                    "labels": p["labels"].cpu(),
                }
                for p in preds
            ]

            processed_targets = [
                {
                    "boxes":  t["boxes"].cpu(),
                    "labels": t["labels"].cpu(),
                }
                for t in targets
            ]

            metric.update(processed_preds, processed_targets)

    return ... # Compute final mAP scores


---

## 4️⃣ Running Training & Validation


In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4,
)

num_epochs = 5

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    map_results = validate(model, val_loader, device)

    print(f"Epoch {epoch+1}/{num_epochs}  |  Loss: {train_loss:.4f}  |  "
          f"mAP@0.5:0.95: {map_results['map']:.4f}  |  "
          f"mAP@0.50: {map_results['map_50']:.4f}")


---

## 5️⃣ Visualising Predictions vs. Ground Truth

- **Red boxes** = Ground Truth  
- **Green boxes** = Model Predictions (score ≥ 0.5)


In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torchvision.transforms.functional as F

# Pick 3 random validation images
sample_indices = random.sample(range(len(val_dataset)), 3)

model.eval()
fig, axes = plt.subplots(1, 3, figsize=(30, 10))

for ax, idx in zip(axes, sample_indices):
    img_tensor, target = val_dataset[idx]

    # Run inference
    with torch.no_grad():
        pred = model([img_tensor.to(device)])[0]

    img_pil = F.to_pil_image(img_tensor)
    ax.imshow(img_pil)

    # ── Ground Truth (red) ─────────────────────────────────────────────
    for box, lbl in zip(target["boxes"].tolist(), target["labels"].tolist()):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor="red", facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(x1, y1 - 4, label_to_name.get(lbl, str(lbl)),
                color="red", fontsize=9,
                bbox=dict(facecolor="white", alpha=0.6, pad=1))

    # ── Predictions (green, confidence ≥ 0.5) ─────────────────────────
    score_mask = pred["scores"] >= 0.5
    for box, lbl, score in zip(
        pred["boxes"][score_mask].cpu().tolist(),
        pred["labels"][score_mask].cpu().tolist(),
        pred["scores"][score_mask].cpu().tolist(),
    ):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor="green", facecolor="none"
        )
        ax.add_patch(rect)
        name = label_to_name.get(lbl - 1, str(lbl))   # model outputs 1-indexed (bg=0)
        ax.text(x1, y2 + 12, f"{name} {score:.2f}",
                color="green", fontsize=9,
                bbox=dict(facecolor="white", alpha=0.6, pad=1))

    ax.axis("off")
    ax.set_title(f"Val sample #{idx}", fontsize=12)

plt.suptitle("Red = Ground Truth  |  Green = Predictions", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("detection_results.png", bbox_inches="tight", dpi=120)
plt.show()
print("Saved to detection_results.png")


---

## 6️⃣ Saving the Fine-Tuned Model


In [ ]:
torch.save(model.state_dict(), "fasterrcnn_coco_finetuned.pth")
print("Model saved to fasterrcnn_coco_finetuned.pth")


---

## 📚 Summary

| Step | What we did |
|---|---|
| **Dataset** | Loaded COCO 2017 val2017 via `CocoDetectionDataset`, converted bbox format and remapped labels |
| **Collate** | Custom `collate_fn` to handle variable-length box lists |
| **Model** | Pretrained Faster R-CNN MobileNetV3-FPN, head swapped for 80 COCO classes |
| **Training** | Backbone frozen; only detection head fine-tuned with AdamW |
| **Evaluation** | mAP@0.5:0.95 via `torchmetrics` |
| **Visualisation** | Overlaid ground-truth (red) and predictions (green) on sample images |

